<a href="https://colab.research.google.com/github/Trang19/6680/blob/main/6680sept16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [4]:
import glob

years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
outage_dfs = []

for year in years:
    # Construct a glob pattern for the StormEvents files
    file_pattern = f'/content/StormEvents_details-ftp_v1.0_d{year}_c*.csv.gz'
    matching_files = glob.glob(file_pattern)

    if not matching_files:
        print(f"Warning: No file found for year {year} matching pattern {file_pattern}")
        continue # Skip to the next year

    # Assuming there's only one matching file or we just take the first one
    filepath = matching_files[0]
    print(f"Loading {filepath}")
    df = pd.read_csv(filepath, compression='gzip') # Specify compression for .gz files
    df['year'] = year
    outage_dfs.append(df)

if outage_dfs: # Only concatenate if any data was loaded
    outages = pd.concat(outage_dfs, ignore_index=True)
    print(f"Loaded {len(outages):,} records from {len(outage_dfs)} years")
else:
    print("No data was loaded for any year.")

Loading /content/StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2017_c20260519.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2018_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2019_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2020_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2021_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2022_c20260625.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2023_c20260323.csv.gz
Loaded 629,141 records from 10 years


In [6]:
import glob

noaa_dfs = []
# Assuming 'years' is defined in a previous cell

for year in years:
    # Construct a glob pattern for the StormEvents files, similar to the previous cell
    file_pattern = f'/content/StormEvents_details-ftp_v1.0_d{year}_c*.csv.gz'
    matching_files = glob.glob(file_pattern)

    if not matching_files:
        print(f"Warning: No file found for year {year} matching pattern {file_pattern}")
        continue # Skip to the next year

    # Assuming there's only one matching file or we just take the first one
    filepath = matching_files[0]
    print(f"Loading {filepath}")
    df = pd.read_csv(filepath, compression='gzip') # Specify compression for .gz files
    df['year'] = year # Add year column for consistency
    noaa_dfs.append(df)

if noaa_dfs: # Only concatenate if any data was loaded
    noaa = pd.concat(noaa_dfs, ignore_index=True)
    print(f"Loaded {len(noaa):,} NOAA storm event records from {len(noaa_dfs)} years")
else:
    print("No NOAA storm event data was loaded for any year.")

Loading /content/StormEvents_details-ftp_v1.0_d2014_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2015_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2016_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2017_c20260519.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2018_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2019_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2020_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2021_c20260323.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2022_c20260625.csv.gz
Loading /content/StormEvents_details-ftp_v1.0_d2023_c20260323.csv.gz
Loaded 629,141 NOAA storm event records from 10 years


In [8]:
noaa['BEGIN_DATE_TIME'] = pd.to_datetime(noaa['BEGIN_DATE_TIME'],
format='%d-%b-%y %H:%M:%S', errors='coerce')
noaa['END_DATE_TIME'] = pd.to_datetime(noaa['END_DATE_TIME'],
format='%d-%b-%y %H:%M:%S', errors='coerce')

In [9]:
relevant_storms = ['Hurricane', 'Tropical Storm', 'Tornado',
'Thunderstorm Wind', 'High Wind', 'Winter Storm',
'Winter Weather', 'Heavy Snow', 'Blizzard',
'Wildfire', 'Flash Flood', 'Flood']
noaa_filtered = noaa[noaa['EVENT_TYPE'].isin(relevant_storms)].copy()
print(f"Filtered to {len(noaa_filtered):,} relevant storm events")

Filtered to 385,559 relevant storm events


In [11]:
storm_chars = noaa_filtered.groupby('CZ_FIPS').agg({
'MAGNITUDE': 'max',
# Peak wind speed
'DAMAGE_PROPERTY': 'count',
# Number of storm events
'INJURIES_DIRECT': 'sum',
'DEATHS_DIRECT': 'sum',
'BEGIN_DATE_TIME': 'min',
'END_DATE_TIME': 'max'
}).rename(columns={
'MAGNITUDE': 'peak_wind_speed',
'DAMAGE_PROPERTY': 'storm_event_count',
'INJURIES_DIRECT': 'total_injuries',
'DEATHS_DIRECT': 'total_deaths',
'BEGIN_DATE_TIME': 'storm_start',
'END_DATE_TIME': 'storm_end'
}).reset_index()

In [12]:
outages['start_time'] = pd.to_datetime(outages['start_time'])
outages['duration'] = pd.to_numeric(outages['duration'], errors='coerce')
outages['max_customers'] = pd.to_numeric(outages['max_customers'], errors='coerce')

KeyError: 'start_time'